# Repeatable job

`RepeatableJob` wraps an `ns.Job` and submits it again when something outside the job stopped an attempt: a cancellation it did not ask for, such as a preemption, or a platform error such as no free capacity. When the job itself fails, it stops, because running the same bug again only costs money.

Rescheduling only helps when the job can continue where it stopped. The worker in this guide, [`resumable_worker.py`](resumable_worker.py), follows three rules:

1. On start, load the last checkpoint from storage that outlives the job (here, a mounted bucket).
2. Save a checkpoint every few steps, and again on SIGTERM.
3. Exit 0 only when the work is done.

Preemption is simulated by cancelling the attempt from outside. Training takes 150 one-second steps, and the first two attempts are cancelled after 60 seconds each, so it needs three attempts.

**Before you start**

- `uv sync` in the repo root, then start Jupyter from this directory, for example `uv run --with jupyter jupyter lab examples/signals`.
- A Nebius CLI profile (`nebius profile create`) and a bucket you can mount read-write. Set `BUCKET_ID` below to its id (`storagebucket-…`). The `s3://name` form is rejected by the API.
- The jobs run on a CPU machine (`cpu-d3`, `2vcpu-8gb`). **Sections 5 and 7 start billable jobs: three, then one.**

In [ ]:
import nebius_serverless as ns

ns.diagnostics()

In [ ]:
import asyncio
from pathlib import Path
from uuid import uuid4

# Bucket id, not name: nebius storage bucket get-by-name --name <bucket>
BUCKET_ID = "storagebucket-e008847641561172892054"  # ez-bucket-1
CKPT_DIR = f"/ckpt/repeatable-job/{uuid4().hex[:8]}"  # fresh per notebook run
PREEMPT_AFTER = 60  # simulated preemption: cancel after this many seconds
TOTAL_STEPS = 150  # one step is one second
WORKER = Path("resumable_worker.py").resolve()  # uploaded into each attempt

CKPT_DIR

## 1. Why did an attempt end?

The final state already says who stopped the job. `state_details.code` adds the specific cause. These are the combinations seen across 1,406 jobs in one project:

| State | `state_details.code` | Reason | Reschedule? |
|---|---|---|---|
| `CANCELLED` | none | `cancelled`: someone called cancel, from outside the job | yes, unless `RepeatableJob` asked for it |
| `FAILED` | `ContainerFailed` | `job`: the container exited non-zero | no |
| `FAILED` | `TimeoutExceeded` | `job`: the job ran past its `timeout` | no |
| `ERROR` | `NotEnoughResources`, `QuotaExceeded`, or none | `platform`: Nebius could not run it | yes |
| `ERROR` | `VMStopped` | `platform`: the VM under the job was stopped or deleted | yes |

The record has no exit code. A worker that exits non-zero after SIGTERM ends `FAILED`, the same as a crash.

`VMStopped` comes from stopping the job's VM directly (`nebius compute instance stop` on `status.instances[0].compute_instance_id`). The container still gets SIGTERM about 4 seconds later and SIGKILL about 9.5 seconds after that, as with cancel. A spot preemption stops the VM too, so it most likely ends the same way, but none has been observed yet. Check `end_reason` on your first preempted run and adjust `REASONS` if needed.

In [ ]:
REASONS = {
    "COMPLETED": "completed",
    "CANCELLED": "cancelled",  # someone called cancel
    "FAILED": "job",  # non-zero exit, or past the job timeout
    "ERROR": "platform",  # no capacity, quota, ...
}


def end_reason(run: ns.Run) -> tuple[str, str]:
    """Return (reason, state_details.code) for a finished run."""
    details = run.record.get("status", {}).get("state_details") or {}
    return REASONS.get(run.status, "unknown"), details.get("code", "")

## 2. The wrapper

Each attempt is a new job created from the same recipe, with its own id and logs. The wrapper only decides whether to submit again. Progress lives in the checkpoint, not in the wrapper.

To stop for good, call `repeatable.cancel_async()`. It cancels the current attempt and marks the cancellation as requested, so it is not rescheduled. Any other `CANCELLED` counts as external.

`Job(retries=N)` asks Nebius to restart the job. `RepeatableJob` does it from the client: the loop is plain Python, so it can change the recipe between attempts, for example to fall back from spot to on-demand.

In [ ]:
class RepeatableJob:
    """Submit a job again when something outside the job stopped it."""

    def __init__(
        self,
        job: ns.Job,
        *,
        max_attempts: int = 3,
        reschedule: tuple[str, ...] = ("cancelled", "platform"),
    ) -> None:
        self.job = job
        self.max_attempts = max_attempts
        self.reschedule = reschedule
        self.runs: list[ns.Run] = []  # one Run per attempt
        self._stopping = False

    async def run_async(self) -> ns.Run:
        for attempt in range(1, self.max_attempts + 1):
            run = await self.job.create_async()
            self.runs.append(run)
            await run.wait_async()
            reason, code = end_reason(run)
            again = reason in self.reschedule and not self._stopping
            print(
                f"attempt {attempt}: {run.status} {code}".rstrip(),
                f"({reason}) -> {'reschedule' if again else 'stop'}",
            )
            if not again:
                return run
        raise RuntimeError(
            f"no attempt completed in {self.max_attempts} tries"
        )

    async def cancel_async(self) -> None:
        """Stop for good: cancel the current attempt, do not reschedule."""
        self._stopping = True
        if self.runs:
            await self.runs[-1].cancel_async()

## 3. Describe the job

- `volumes` mounts the bucket by id at `/ckpt`, so the checkpoint survives between attempts.
- `inject_files` uploads the worker into the container. `WORKER` is resolved from the notebook directory.
- `env` tells the worker where its checkpoint is and how many steps to run.

In [ ]:
job = ns.Job(
    ["python", "-u", "/app/resumable_worker.py"],
    image="python:3.12-slim",
    timeout="1h",
    volumes=[f"{BUCKET_ID}:/ckpt:rw"],
    inject_files=[f"{WORKER}:/app/resumable_worker.py"],
    env={"CKPT_DIR": CKPT_DIR, "TOTAL_STEPS": str(TOTAL_STEPS)},
    name="repeatable-job",
)

## 4. Simulate preemption

`simulate_preemption` cancels the first `times` attempts, each `after` seconds after it starts running. It uses its own handle on the job, so to `RepeatableJob` this is an external cancellation.

Cancel sends SIGTERM to the container's main process about 4–5 seconds after the request is accepted, and SIGKILL about 9 seconds after SIGTERM (measured on two CPU jobs). The worker saves its exact step well inside that window. For real training, the current step plus the checkpoint write must fit in those ~9 seconds; if they don't, the next attempt resumes from the last periodic checkpoint. The `SIGTERM:` lines in section 6 show where each attempt stopped.

In [ ]:
WAITING = ("PROVISIONING", "STARTING", "IMAGE_PULLING")


async def simulate_preemption(
    repeatable: RepeatableJob, *, after: float, times: int
) -> None:
    """Cancel the first `times` attempts, `after` seconds into RUNNING."""
    for n in range(times):
        while len(repeatable.runs) <= n:  # wait for attempt n + 1
            await asyncio.sleep(5)
        probe = await ns.jobs.get_async(repeatable.runs[n].id, quiet=True)
        while probe.status in WAITING:
            await asyncio.sleep(5)
            await probe.get_async(quiet=True)
        if probe.status == "RUNNING":
            await asyncio.sleep(after)
            await probe.cancel_async(quiet=True)
            print(f"simulated preemption: cancelled {probe.id}")

## 5. Run until complete

Expect three attempts: the first two end `CANCELLED` and are rescheduled, the third completes. Each attempt waits a minute or two for a machine.

Interrupting this cell stops the loop but not the current attempt. Cancel it with `await repeatable.cancel_async()`.

In [ ]:
repeatable = RepeatableJob(job, max_attempts=5)
run, _ = await asyncio.gather(
    repeatable.run_async(),
    simulate_preemption(repeatable, after=PREEMPT_AFTER, times=2),
)
run.display()

## 6. Check the attempts

Every attempt should start at the step where the previous one saved its checkpoint.

In [ ]:
for n, r in enumerate(repeatable.runs, start=1):
    reason, code = end_reason(r)
    print(f"attempt {n}: {r.id} {r.status} {code}".rstrip(), f"({reason})")
    !nebius ai job logs {r.id} | grep -E "starting|SIGTERM|done"

## 7. A job problem is not rescheduled

This job exits non-zero on purpose. It ends `FAILED` with `ContainerFailed`, which `end_reason` reports as `job`, so `RepeatableJob` stops after one attempt even though it is allowed three.

In [ ]:
broken = ns.Job(
    ["python", "-c", "raise SystemExit('bug in the job')"],
    image="python:3.12-slim",
    timeout="1h",
    name="repeatable-job-broken",
)
failed = await RepeatableJob(broken, max_attempts=3).run_async()
end_reason(failed)